# Qwen3.5-0.8B-Base 快速上手

本 notebook 演示三种最小用法：

1. 加载模型（`AutoModelForImageTextToText`，0.853B，bf16）
2. 纯文本续写（Base 模型只做 next-token 预测，不会回答问题）
3. 按官方 chat template 组织输入看看效果差异

**环境**：`qwen-gpu` 内核（`/home/dev/venv`），transformers ≥ 5.20，CUDA 可用。

| 项 | 值 |
| --- | --- |
| 权重路径 | `/home/dev/models/Qwen3.5-0.8B-Base` |
| 参数量 | 873.4M（推理加载 853M，MTP 头被跳过） |
| 上下文 | 262,144 tokens |
| 显存 | bf16 约 1.7 GB |

In [1]:
import os
import torch

MODEL_PATH = "/home/dev/models/Qwen3.5-0.8B-Base"
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE.startswith("cuda") else torch.float32

print("torch", torch.__version__)
print("device", DEVICE, torch.cuda.get_device_name(0) if DEVICE.startswith("cuda") else "")
print("model dir exists:", os.path.isdir(MODEL_PATH))

torch 2.14.1+cu126
device cuda:0 NVIDIA L20
model dir exists: True


## 1. 加载

`AutoModelForImageTextToText` 对应 `Qwen3_5ForConditionalGeneration`，会同时加载视觉塔。
只做纯文本实验时也可以用 `AutoModelForCausalLM`（`Qwen3_5ForCausalLM`，752M），更省显存。

In [2]:
import transformers
from transformers import AutoTokenizer, AutoModelForImageTextToText

print("transformers", transformers.__version__)

tok = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH, dtype=DTYPE, device_map=DEVICE
).eval()

n = sum(p.numel() for p in model.parameters())
print(type(model).__name__, f"{n/1e9:.3f}B params")
if DEVICE.startswith("cuda"):
    print(f"显存占用 {torch.cuda.memory_allocated()/1e9:.2f} GB")

/home/dev/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers 5.20.2


Current Python version 3.10 is below the recommended 3.11 version. It is recommended to upgrade to Python 3.11 or higher for the best experience.



Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


Loading weights:  35%|███▌      | 167/473 [00:00<00:00, 1656.95it/s]


Loading weights: 100%|██████████| 473/473 [00:00<00:00, 2451.80it/s]

Qwen3_5ForConditionalGeneration 0.853B params
显存占用 1.71 GB


## 2. 纯文本续写

Base 模型没有指令对齐，`generate` 只会顺着上文往下写。

In [3]:
def continue_text(prompt: str, max_new_tokens: int = 40) -> str:
    ids = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True)


for p in [
    "The capital of France is",
    "def fibonacci(n):\n    \"\"\"Return the n-th Fibonacci number.\"\"\"\n",
    "人工智能的核心思想是",
]:
    print("PROMPT:", repr(p))
    print("OUT   :", repr(continue_text(p)))
    print("-" * 60)

PROMPT: 'The capital of France is'


OUT   : ' located in the northwestern part of the country. In the north, the capital is the city of Paris. In the east, the capital is the city of Nantes. In the south, the capital'
------------------------------------------------------------
PROMPT: 'def fibonacci(n):\n    """Return the n-th Fibonacci number."""\n'


OUT   : '    if n == 0:\n        return 0\n    if n == 1:\n        return 1\n    return fibonacci(n - 1) + fibonacci(n - 2'
------------------------------------------------------------
PROMPT: '人工智能的核心思想是'


OUT   : '（）\nA. 智能化\nB. 智能化、自动化\nC. 智能化、自动化、智能化\nD. 智能化、自动化、智能化\n\n<think>\n我们被问到："人工智能'
------------------------------------------------------------


## 3. 贪心解码 + logits 查看

不采样，直接看模型最确信的下一个 token。

In [4]:
ids = tok("The capital of France is", return_tensors="pt").to(model.device)
with torch.no_grad():
    logits = model(**ids, use_cache=False).logits[0, -1].float()

probs = logits.softmax(-1)
top = probs.topk(5)
for p, i in zip(top.values.tolist(), top.indices.tolist()):
    print(f"{p:7.2%}  {tok.decode([i])!r}")

 26.33%  ' Paris'
  8.03%  ' the'
  6.25%  ' ______'
  6.25%  ':'
  4.58%  '\n'


## 4. 套用官方 chat template

`tokenizer_config.json` 里内置了 Qwen3.5 的 chat template（含 `<|im_start|>` / `<|im_end|>`）。
Base 权重没有经过指令对齐，套模板通常不会得到像样的回答——这里只是展示模板产物长什么样，
以及为什么做 LoRA 时可以直接复用它（`<|im_start|>` 等控制 token 在预训练阶段已训练过）。

In [5]:
msgs = [{"role": "user", "content": "用一句话解释什么是大语言模型。"}]
text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
print(text)

ids = tok(text, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(
        **ids, max_new_tokens=60, do_sample=True, temperature=0.7,
        top_p=0.9, pad_token_id=tok.eos_token_id,
    )
print("=" * 60)
print(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True))

<|im_start|>user
用一句话解释什么是大语言模型。<|im_end|>
<|im_start|>assistant
<think>

</think>




大语言模型（Large Language Model）是一种基于深度学习的计算机模型，能够处理自然语言，生成人类可理解的文字。



## 5. 实测：Countdown / 1D-ARC —— 模型回答 vs 正确答案、正确率

直接在代码里从两个数据集**随机抽样**（固定种子，不落盘中间文件）。判定全部程序化：

- **Countdown**（`data/countdown/data/train-*.parquet`，`{target, nums}`）：从回答里抽取算式，要求**恰好用完给定数字**（多重集合相同），求值与 target 比对；正确答案由程序暴力搜索。
- **1D-ARC**（`data/1d_arc/1D-ARC-main/dataset/*/*.json`，演示对 + test 查询）：解析与网格等宽的整数行，与 ground-truth 网格**精确匹配**（另附逐格准确率供参考，注意抄输入就能拿高分）。

5.1 先看 2+2 题的完整回答对照，5.2 每个数据集抽 20 题测正确率。

In [6]:
import glob, json, os, random, re
from fractions import Fraction

COUNTDOWN = '/home/dev/KAP/data/countdown/data/train-00000-of-00001.parquet'
ARC_DIR = '/home/dev/KAP/data/1d_arc/1D-ARC-main/dataset'

def sample_countdown(n, seed):
    import pyarrow.parquet as pq
    t = pq.read_table(COUNTDOWN)
    rng = random.Random(seed)
    out = []
    for i in rng.sample(range(t.num_rows), n):
        tgt = t.column('target')[i].as_py()
        nums = t.column('nums')[i].as_py()
        prompt = (f'Using the numbers {nums}, create an expression that equals {tgt}. '
                  'You can use basic arithmetic operations (+, -, *, /). '
                  'Each number must be used exactly once.')
        out.append({'dataset': 'countdown', 'id': f'row{i}', 'prompt': prompt,
                    'raw': {'nums': nums, 'target': tgt}})
    return out

def sample_arc(n, seed):
    files = sorted(glob.glob(f'{ARC_DIR}/*/*.json'))
    rng = random.Random(seed)
    out = []
    for f in rng.sample(files, n):
        task = json.load(open(f))
        demos, test = task['train'], task['test'][0]
        row = lambda g: ' '.join(map(str, g[0]))
        parts = ['Apply the same transformation rule to the test input.',
                 'Here are example input/output pairs:']
        for dmo in demos:
            parts += ['Input:', row(dmo['input']), 'Output:', row(dmo['output'])]
        parts += ['Now, give the output for the test input.', 'Test Input:', row(test['input']), 'Output:']
        out.append({'dataset': '1d_arc', 'id': '/'.join(f.split('/')[-2:]),
                    'prompt': chr(10).join(parts), 'raw': {'test': test}})
    return out

def countdown_truth(nums, target):
    """暴力搜索一个恰好用完 nums 且等于 target 的算式，返回表达式字符串或 None。"""
    def gen(items):
        if len(items) == 1:
            yield items[0]
            return
        n = len(items)
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                x, sx = items[i]
                y, sy = items[j]
                others = [items[k] for k in range(n) if k != i and k != j]
                for v, s in ((x + y, f'({sx}+{sy})'), (x - y, f'({sx}-{sy})'),
                             (x * y, f'({sx}*{sy})')) + (((x / y, f'({sx}/{sy})'),) if y != 0 else ()):
                    yield from gen(others + [(v, s)])
    for v, s in gen([(Fraction(n_), str(n_)) for n_ in nums]):
        if v == target:
            return s
    return None

def check_countdown(resp, nums, target):
    """-> (correct: bool, detail: str)。正确 = 存在恰好用齐数字且值为 target 的算式。"""
    cands = re.findall(r'[\d\(\)\+\-\*\/][\d\(\)\+\-\*\/\s]*[\d\)]', resp)
    tried = []
    for c in cands:
        if not re.search(r'[\+\-\*\/]', c):
            continue
        runs = re.findall(r'\d+', c)
        if sorted(map(int, runs)) != sorted(nums):
            continue
        try:
            val = eval(c, {'__builtins__': {}}, {})
        except Exception:
            continue
        if isinstance(val, float) and abs(val - round(val)) < 1e-9:
            val = round(val)
        if isinstance(val, int):
            tried.append((re.sub(r'\s+', '', c), val))
    for expr, val in tried:
        if val == target:
            return True, f'{expr} = {val}'
    if tried:
        return False, '用齐数字但结果错: ' + '; '.join(f'{e}={v}' for e, v in tried[:2]) + f' (target {target})'
    return False, '未找到恰好用齐数字的算式'

def check_arc(resp, test_out):
    """-> (correct: bool, detail: str)。正确 = 解析到等宽行且与 truth 精确一致。"""
    truth = test_out[0]
    width = len(truth)
    for line in resp.splitlines():
        xs = re.findall(r'\d+', line)
        if len(xs) == width:
            cand = list(map(int, xs))
            if cand == truth:
                return True, '精确匹配'
            acc = sum(a == b for a, b in zip(cand, truth)) / width
            return False, f'不匹配（逐格 {acc:.0%}，抄输入也会高，须精确匹配）'
    return False, f'未解析到宽度 {width} 的整数行'

print('samplers ready')

samplers ready


In [7]:
def gen_response(prompt, max_new_tokens=512):
    ids = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, top_p=0.9,
                             eos_token_id=[tok.eos_token_id, tok.convert_tokens_to_ids('<|im_end|>')],
                             pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

# ---- 5.1 四题完整对照 ----
for it in sample_countdown(2, 0) + sample_arc(2, 0):
    resp = gen_response(it['prompt'])
    print('=' * 76)
    print(f"[{it['dataset']}] {it['id']}")
    print('PROMPT  :', it['prompt'][:150].replace('\n', ' | '), '...' if len(it['prompt']) > 150 else '')
    print('RESPONSE:', resp[:400].replace('\n', ' | '), '...' if len(resp) > 400 else '')
    if it['dataset'] == 'countdown':
        nums, target = it['raw']['nums'], it['raw']['target']
        print('正确示例:', countdown_truth(nums, target), '=', target)
        ok, detail = check_countdown(resp, nums, target)
        print('判定    :', ('正确 ' if ok else '错误 ') + detail)
    else:
        ok, detail = check_arc(resp, it['raw']['test']['output'])
        print('正确输出:', ' '.join(map(str, it['raw']['test']['output'][0])))
        print('判定    :', ('正确 ' if ok else '错误 ') + detail)

[countdown] row442720
PROMPT  : Using the numbers [37, 15, 4, 13], create an expression that equals 16. You can use basic arithmetic operations (+, -, *, /). Each number must be used ...
RESPONSE:  |  | To solve this, let's break it down step by step. We need to use the numbers 37, 15, 4, and 13 exactly once to get an answer of 16. One way to do this is by using the equation \(37 - 15 \times 4 + 13\). Let's calculate this step by step: |  | 1. First, multiply 15 by 4: \(15 \times 4 = 60\). | 2. Then, subtract 60 from 37: \(37 - 60 = -23\). | 3. Finally, add 13 to -23: \(-23 + 13 = -10\). |  | This gives us ...
正确示例: ((37+15)*(4/13)) = 16
判定    : 错误 未找到恰好用齐数字的算式


[countdown] row201979
PROMPT  : Using the numbers [3, 5, 39, 31], create an expression that equals 29. You can use basic arithmetic operations (+, -, *, /). Each number must be used  ...
RESPONSE:  |  | To create an expression that equals 29 using the numbers [3, 5, 39, 31], we need to combine them with basic arithmetic operations. One possible solution is: |  | \[ 39 + 5 - 3 \times 31 \] |  | Let's break it down step by step: | 1. First, multiply 3 and 31: \( 3 \times 31 = 93 \). | 2. Next, subtract 93 from 5: \( 5 - 93 = -88 \). | 3. Finally, add 39 to the result: \( 39 + (-88) = -49 \). |  | This gives us -49 ...
正确示例: (5+(3*(39-31))) = 29
判定    : 错误 用齐数字但结果错: 39+5+31-3=72 (target 29)
[1d_arc] 1d_scale_dp/1d_scale_dp_21.json
PROMPT  : Apply the same transformation rule to the test input. | Here are example input/output pairs: | Input: | 0 0 0 0 0 7 7 7 7 7 7 7 0 0 8 0 | Output: | 0 0 0 0 0 7  ...
RESPONSE:  
正确输出: 1 1 1 1 1 1 1 1 1 8 0 0 0 0 0 0
判定    : 错误 未解析到宽度 16 的整数行
[1d_arc] 1d

In [8]:
# ---- 5.2 正确率：每个数据集 20 题（种子不同，与 5.1 不重叠不保证）----
N = 20
summary = {}
for name, sampler in (('countdown', sample_countdown), ('1d_arc', sample_arc)):
    items = sampler(N, 123)
    ok_n, lines = 0, []
    for k, it in enumerate(items):
        resp = gen_response(it['prompt'])
        if it['dataset'] == 'countdown':
            ok, detail = check_countdown(resp, it['raw']['nums'], it['raw']['target'])
        else:
            ok, detail = check_arc(resp, it['raw']['test']['output'])
        ok_n += ok
        lines.append(f"  {'V' if ok else 'X'} {it['id'][:28]:28s} {detail[:70]}")
        if (k + 1) % 10 == 0:
            print(f'  ... {name} {k + 1}/{N}', flush=True)
    summary[name] = ok_n
    print(f'\n[{name}] 正确率 {ok_n}/{N} = {ok_n / N:.0%}')
    print('\n'.join(lines))
    print()
print('汇总:', {k: f'{v}/{N} = {v / N:.0%}' for k, v in summary.items()})

  ... countdown 10/20


  ... countdown 20/20



[countdown] 正确率 3/20 = 15%
  X row27453                     未找到恰好用齐数字的算式
  V row140339                    78+47-40 = 85
  X row45710                     未找到恰好用齐数字的算式
  X row403154                    未找到恰好用齐数字的算式
  X row213511                    未找到恰好用齐数字的算式
  X row139750                    未找到恰好用齐数字的算式
  X row56465                     未找到恰好用齐数字的算式
  X row439617                    用齐数字但结果错: (71-25)*15=690 (target 61)
  X row472487                    未找到恰好用齐数字的算式
  X row458660                    用齐数字但结果错: 79+34-77=36 (target 17)
  V row20003                     18+12+40 = 70
  X row198770                    未找到恰好用齐数字的算式
  X row281124                    未找到恰好用齐数字的算式
  X row294816                    未找到恰好用齐数字的算式
  V row174167                    67-16-8 = 43
  X row178676                    未找到恰好用齐数字的算式
  X row446738                    未找到恰好用齐数字的算式
  X row27211                     未找到恰好用齐数字的算式
  X row83709                     未找到恰好用齐数字的算式
  X row70809                     未找到恰好用齐数字的算式



  ... 1d_arc 10/20


  ... 1d_arc 20/20



[1d_arc] 正确率 0/20 = 0%
  X 1d_denoising_mc/1d_denoising 未解析到宽度 33 的整数行
  X 1d_mirror/1d_mirror_30.json  未解析到宽度 25 的整数行
  X 1d_denoising_mc/1d_denoising 未解析到宽度 32 的整数行
  X 1d_recolor_cnt/1d_recolor_cn 未解析到宽度 19 的整数行
  X 1d_move_2p_dp/1d_move_2p_dp_ 未解析到宽度 26 的整数行
  X 1d_mirror/1d_mirror_29.json  未解析到宽度 17 的整数行
  X 1d_fill/1d_fill_18.json      不匹配（逐格 34%，抄输入也会高，须精确匹配）
  X 1d_scale_dp/1d_scale_dp_16.j 未解析到宽度 22 的整数行
  X 1d_scale_dp/1d_scale_dp_5.js 未解析到宽度 25 的整数行
  X 1d_denoising_1c/1d_denoising 未解析到宽度 33 的整数行
  X 1d_move_2p/1d_move_2p_43.jso 不匹配（逐格 20%，抄输入也会高，须精确匹配）
  X 1d_move_dp/1d_move_dp_9.json 未解析到宽度 25 的整数行
  X 1d_padded_fill/1d_padded_fil 未解析到宽度 87 的整数行
  X 1d_move_1p/1d_move_1p_45.jso 不匹配（逐格 88%，抄输入也会高，须精确匹配）
  X 1d_move_1p/1d_move_1p_8.json 未解析到宽度 9 的整数行
  X 1d_scale_dp/1d_scale_dp_29.j 未解析到宽度 10 的整数行
  X 1d_flip/1d_flip_20.json      未解析到宽度 29 的整数行
  X 1d_fill/1d_fill_43.json      未解析到宽度 11 的整数行
  X 1d_move_1p/1d_move_1p_5.json 未解析到宽度 25 的整数行
  X 1d_padded_fill/1d_padded_fil 未解

### 5.3 改用 DSpark 快速引擎生成（Gumbel-max 单 uniform 采样）

- 采样改为：**每个选择一个 [0,1] uniform**——草稿 7 步用预抽取的 Gumbel 噪声逐槽 argmax（逆 Gumbel CDF），接受判定与修正 token 融合成每槽**一个** uniform 的逆 CDF（u < a 接受；否则 u 的剩余段在 (p−q)⁺ 的 CDF 上查表得到修正 token）。
- 无损性不受影响（同一分布的另一种精确采样器）；对照 HF 纯 T=0.7（无 top_p，与 DSpark 同参）的正确率应一致。
- drafter：v2-B best。

In [9]:
import sys, time
sys.path.insert(0, '/home/dev/KAP/src4dspark')
from dspark_paper import DSparkDrafter, ND
import dspark_fast

d = DSparkDrafter(model, n_draft=ND, layers=5, markov_rank=128, device=model.device, dtype=torch.float32)
ck = torch.load('/home/dev/KAP/parameter/draft/dspark_pb_v2_b/best.pt', map_location=model.device, weights_only=False)
d.load_state_dict(ck['model']); d.eval()
eng = dspark_fast.DSparkDecodeFast(model, d, tok, temperature=0.7, seed=123)
print(f"drafter dspark_pb_v2_b/best.pt  teacher-forced tau {ck['tau']:.3f}")

def gen_hf07(prompt, max_new_tokens=512):
    ids = tok(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True,
                             temperature=0.7, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True)

summary53 = {}
for name, sampler in (('countdown', sample_countdown), ('1d_arc', sample_arc)):
    items = sampler(N, 123)
    acc = {}
    for mode in ('HF T=0.7', 'DSpark'):
        ok_n, spd = 0, []
        for it in items:
            torch.cuda.synchronize(); t0 = time.perf_counter()
            if mode == 'HF T=0.7':
                resp = gen_hf07(it['prompt']); ntok = 0
            else:
                ids_out, st = eng.generate(tok(it['prompt'])['input_ids'], 512)
                resp = tok.decode(ids_out, skip_special_tokens=True); ntok = st['tokens']
            torch.cuda.synchronize(); dt = time.perf_counter() - t0
            if ntok:
                spd.append(ntok / dt)
            if it['dataset'] == 'countdown':
                ok, _ = check_countdown(resp, it['raw']['nums'], it['raw']['target'])
            else:
                ok, _ = check_arc(resp, it['raw']['test']['output'])
            ok_n += ok
        acc[mode] = (ok_n, sum(spd) / len(spd) if spd else None)
        line = f"[{name}] {mode}: 正确率 {ok_n}/{N} = {ok_n / N:.0%}"
        if spd:
            line += f"   平均 {acc[mode][1]:.1f} tok/s"
        print(line, flush=True)
    summary53[name] = acc

# 一条 DSpark 生成的完整样例
it = sample_countdown(1, 999)[0]
ids_out, st = eng.generate(tok(it['prompt'])['input_ids'], 512)
print('\n样例 [countdown]', it['id'], ' target', it['raw']['target'])
print('DSpark RESPONSE:', tok.decode(ids_out, skip_special_tokens=True)[:400].replace('\n', ' | '))
ok, detail = check_countdown(tok.decode(ids_out, skip_special_tokens=True), it['raw']['nums'], it['raw']['target'])
print('判定:', ok, detail, f"| tau {st['tokens'] / st['rounds']:.2f}, rounds {st['rounds']}, forwards {st['forwards']}")

drafter dspark_pb_v2_b/best.pt  teacher-forced tau 2.159


[countdown] HF T=0.7: 正确率 1/20 = 5%


[countdown] DSpark: 正确率 0/20 = 0%   平均 74.9 tok/s


[1d_arc] HF T=0.7: 正确率 0/20 = 0%


[1d_arc] DSpark: 正确率 0/20 = 0%   平均 27.8 tok/s



样例 [countdown] row409650  target 65
DSpark RESPONSE:  |  | To solve this, we need to find an expression that uses each number [71, 66, 72] exactly once and results in 65. Let's try a few combinations: |  | 1. \( 72 - 71 + 66 = 71 \) (Not 65) | 2. \( 72 + 66 - 71 = 67 \) (Not 65) | 3. \( 71 + 66 + 72 - 72 = 71 \) (Not 65) | 4. \( 71 + 66 + 72 - 72 = 71 \) (Not 65) | 5. \( 71 - 66 + 72 = 77 \) (Not 65) | 6. \( 71 - 66 * 72 = 71 - 4752 = -4681 \) (Not 65) | 7. \( 71 * 66 
判定: False 未找到恰好用齐数字的算式 | tau 3.29, rounds 156, forwards 156


### 5.4 Gumbel-max 机制 + top-K draft 块概率（演示）

两部分：① 用 20 万次采样验证 Gumbel-max 的精确性（`argmax(log p + G)`，`G=−log(−log u)` 就是 Gumbel 的逆 CDF）；② 由于 markov 头使 drafter 成为一阶链，**beam search 7 步即可拿到概率最大的 K 个 draft 块及其精确概率**（P(draft)=各槽条件概率连乘），再对每块算 P(全部接受)=Π αⱼ。

In [ ]:
import sys, math, torch
sys.path.insert(0, '/home/dev/KAP/src4dspark')
from dspark_paper import DSparkDrafter, ND
import dspark_fast

d = DSparkDrafter(model, n_draft=ND, layers=5, markov_rank=128, device=model.device, dtype=torch.float32)
ck = torch.load('/home/dev/KAP/parameter/draft/dspark_pb_v2_b/best.pt', map_location=model.device, weights_only=False)
d.load_state_dict(ck['model']); d.eval()
eng = dspark_fast.DSparkDecodeFast(model, d, tok, temperature=1.0, seed=0)

# ---- 1) toy: Gumbel-max 是否精确 ----
p = torch.tensor([0.5, 0.3, 0.15, 0.05], device=model.device)
g = -torch.log(-torch.log(torch.rand(200000, 4, device=model.device).clamp_min(1e-20)))
freq = (p.log() + g).argmax(-1).bincount(minlength=4) / 200000
print('target p:', [round(x, 3) for x in p.tolist()])
print('gumbel  :', [round(x, 4) for x in freq.tolist()])

# ---- 2) top-K draft 块及概率 ----
prompt = tok('The theory of relativity states that', return_tensors='pt')['input_ids'].to(model.device)
with torch.inference_mode():
    cache, last, feats = eng.prefill(prompt)
pos = prompt.shape[1]
anchor = eng._sample(last)
with torch.inference_mode():
    hctx = d.encode_ctx(feats[:pos].unsqueeze(0))
    o = torch.tensor([[pos - 1]], device=model.device)
    z = d.backbone(hctx, d.embed(anchor.view(1, 1)), o)[0, 0]
    base = d.lm_head(z.to(d.lm_head.weight.dtype)).float()      # 每槽 base logits (7, V)

K = 5
beams = [(0.0, int(anchor), ())]
for j in range(ND):                                             # 一阶链上做 beam search
    cand = []
    for lp, prev, seq in beams:
        lg = (base[j] + d.markov(torch.tensor([prev], device=model.device))[0]) / 1.0
        lsm = torch.log_softmax(lg, -1)
        tv, ti = lsm.topk(K)
        for v, i in zip(tv.tolist(), ti.tolist()):
            cand.append((lp + v, i, seq + (i,)))
    cand.sort(key=lambda x: -x[0])
    beams = cand[:K]

print(f'top-{K} draft blocks (anchor={tok.decode([anchor])!r}):')
for r, (lp, _, seq) in enumerate(beams):
    with torch.inference_mode():
        chunk = torch.tensor([int(anchor)] + list(seq), device=model.device)
        out = model(input_ids=chunk.view(1, -1), position_ids=dspark_fast._pos(pos, ND + 1, model.device),
                    use_cache=False, logits_to_keep=ND + 1)
        vlg = out.logits[0].float()
        alphas, prev_t = [], int(anchor)
        for j in range(ND):
            qj = torch.softmax(base[j] + d.markov(torch.tensor([prev_t], device=model.device))[0], -1)
            pj = torch.softmax(vlg[j], -1)
            alphas.append(float(torch.minimum(pj, qj).sum()))
            prev_t = seq[j]
    print(f'  #{r+1}: P(draft)={math.exp(lp):.2e}  log10P={lp / math.log(10):.1f}  '
          f'P(全接受)={math.prod(alphas):.4f}  text={tok.decode(list(seq))!r}'[:130])

### 5.5 分层嵌套逆 CDF 的玩具演示（机制逐字对应真实实现）

2-slot 块分布：slot1 = [0.6, 0.3, 0.1]（token a/b/c），slot2 依条件。**把所有块按字典序铺在 [0,1)，区间宽度=块概率**；一条路径由一个 uniform 经嵌套逆 CDF 完全决定（每步 searchsorted 后把 u 投影进所选 token 的子区间并重新归一化）。4 分支用分层 uniform `u_b=(b−1+v_b)/4` —— 路径按概率比例铺满分布。最后用 20 万次采样验证边际精确性（无损性的根基）。

In [ ]:
import torch
torch.manual_seed(0)
q1 = torch.tensor([0.6, 0.3, 0.1])                       # slot1: a,b,c —— 对应 DSpark 的 base[0]+markov
q2 = {0: [0.5, 0.5], 1: [0.9, 0.1], 2: [0.2, 0.8]}       # slot2 条件分布 —— 对应 base[1]+markov(x1)
blocks = {(i, j): float(q1[i]) * float(q2[i][j]) for i in range(3) for j in range(2)}
cum, edges = 0.0, {}
for k, p in sorted(blocks.items()):
    edges[k] = (cum, cum + p); cum += p
print('块 -> [0,1) 区间（宽度=块概率）:')
for k, (lo, hi) in edges.items(): print(f'   {k}: [{lo:.2f},{hi:.2f})  p={hi-lo:.2f}')

def chain_inverse(u):                                    # 就是 dspark_fast._propose_multi(mode=stratified) 的内层
    c1 = torch.cumsum(q1, 0)
    x1 = int(torch.searchsorted(c1, torch.tensor(float(u)), right=True).clamp(max=2))
    u1 = (u - float(c1[x1] - q1[x1])) / float(q1[x1])    # 投影进子区间，重新归一化
    c2 = torch.cumsum(torch.tensor(q2[x1]), 0)
    x2 = int(torch.searchsorted(c2, torch.tensor(float(u1)), right=True).clamp(max=1))
    return (x1, x2)

v = torch.rand(4); u4 = (torch.arange(4) + v) / 4        # 分层 uniform（耦合随机数）
paths = [chain_inverse(float(u)) for u in u4]
print('分层 4 分支: u =', [round(float(x), 3) for x in u4])
print('            路径 =', paths)
N = 200000
freq = {}
for t in torch.rand(N).tolist():
    pb = chain_inverse(t); freq[pb] = freq.get(pb, 0) + 1
print('20 万次采样的频率 vs 精确概率（边际精确 => 无损性成立）:')
for k in sorted(blocks): print(f'   {k}: freq {freq.get(k,0)/N:.4f}  exact {blocks[k]:.4f}')

## 6. 清理

In [10]:
import gc

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"剩余显存占用 {torch.cuda.memory_allocated()/1e9:.2f} GB")

剩余显存占用 3.06 GB
